# RFM–LRFM Comparison and Candidate Customer Profiles

This notebook reproduces the preliminary comparison and profiles the four-group RFM candidate. It does not select the final segmentation or assign final perks.

**Input:** `data/own_processed/user_features_windowed.csv` from the study export.

**Outputs:** separate comparison, coverage, assignment and behavioral-profile files in `reports/rfm_lrfm_candidate_profiles/`. Raw data and the source feature table are never overwritten. Repeated runs replace only this notebook's derived outputs.

The full cohort is retained. Customers without complete modeling inputs have explicit pending statuses, not inferred traveler identities. Approximately three final segments may be highlighted in the presentation, but all final segments must be accounted for.

Dependencies: pandas and NumPy. No additional packages or database connection are required.


In [ ]:
from pathlib import Path
import numpy as np,pandas as pd,json
print("STEP 1: Locate the study feature table", flush=True)
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "own_processed" / "user_features_windowed.csv").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("Expected data/own_processed/user_features_windowed.csv beneath the project folder. No output written.")
source = PROJECT / "data" / "own_processed" / "user_features_windowed.csv"
ROOT = PROJECT / "reports" / "rfm_lrfm_candidate_profiles"
ROOT.mkdir(parents=True, exist_ok=True)
print("Input:", source, flush=True)
d=pd.read_csv(source,dtype={'user_id':'string'})
cols=['rfm_recency_days','rfm_frequency_trips','rfm_monetary_estimate','lrfm_length_days']
ready=d.baseline_status.eq('Ready for RFM/LRFM comparison')
m=d.loc[ready].copy();assert len(m)>1000 and d.user_id.is_unique
assert m[cols].notna().all().all() and np.isfinite(m[cols]).all().all()
assert (m.rfm_frequency_trips>0).all()
assert np.allclose(m.rfm_monetary_estimate,m.recent_flight_spend_estimate+m.recent_hotel_spend_estimate)
for c in ['birthdate','sign_up_date','first_valid_booking_date','last_valid_booking_date','analysis_reference_date','feature_window_start','feature_window_end']:
 assert not (d[c].notna()&pd.to_datetime(d[c],errors='coerce').isna()).any()
Z=np.log1p(m[cols].to_numpy(float)); X=(Z-Z.mean(axis=0))/Z.std(axis=0)
assert np.isfinite(X).all()
# Standard Lloyd iterations and k-means++ initialization; no unavailable sklearn dependency.
def fit_once(x,k,rng,max_iter=300):
 n=len(x); centers=[x[rng.integers(n)].copy()]
 best=np.sum((x-centers[0])**2,axis=1)
 for j in range(1,k):
  idx=rng.choice(n,p=best/best.sum()) if best.sum()>0 else rng.integers(n)
  centers.append(x[idx].copy());best=np.minimum(best,np.sum((x-centers[-1])**2,axis=1))
 centers=np.array(centers);previous=None
 for it in range(max_iter):
  dist=np.sum((x[:,None,:]-centers[None,:,:])**2,axis=2);lab=dist.argmin(axis=1)
  if previous is not None and np.array_equal(lab,previous):break
  previous=lab.copy()
  for j in range(k):
   if np.any(lab==j):centers[j]=x[lab==j].mean(axis=0)
   else: centers[j]=x[np.argmax(dist.min(axis=1))]
 inertia=float(np.sum((x-centers[lab])**2));return lab,centers,inertia
def fit(x,k,seed):
 rng=np.random.default_rng(seed);best=None
 for _ in range(10):
  result=fit_once(x,k,rng)
  if best is None or result[2]<best[2]:best=result
 return best
# Agreement invariant to arbitrary label numbering.
def ari(a,b):
 _,ai=np.unique(a,return_inverse=True);_,bi=np.unique(b,return_inverse=True)
 mat=np.zeros((ai.max()+1,bi.max()+1),dtype=np.int64);np.add.at(mat,(ai,bi),1)
 comb=lambda arr:float(np.sum(arr*(arr-1)/2))
 observed=comb(mat);ar=comb(mat.sum(axis=1));br=comb(mat.sum(axis=0));pairs=len(a)*(len(a)-1)/2
 expected=ar*br/pairs;den=(ar+br)/2-expected
 return (observed-expected)/den if den else 1.0
sample=np.random.default_rng(2026).choice(len(m),1000,replace=False)
def sil(x,labels):
 xx=x[sample];ll=labels[sample];dist=np.sqrt(np.sum((xx[:,None,:]-xx[None,:,:])**2,axis=2))
 aa=np.zeros(len(xx));bb=np.full(len(xx),np.inf);single=np.zeros(len(xx),bool)
 for c in np.unique(ll):
  ix=np.where(ll==c)[0];outside=np.where(ll!=c)[0]
  single[ix]=len(ix)==1
  if len(ix)>1:aa[ix]=dist[np.ix_(ix,ix)].sum(axis=1)/(len(ix)-1)
  bb[outside]=np.minimum(bb[outside],dist[np.ix_(outside,ix)].mean(axis=1))
 vals=(bb-aa)/np.maximum(aa,bb);vals[single]=0
 return float(np.nanmean(vals))
# Sanity tests for core numerical functions.
assert abs(ari([0,0,1,1],[1,1,0,0])-1)<1e-12
labels_test,centers_test,objective=fit(np.array([[0.,0.],[.1,0.],[10.,10.],[10.1,10.]]),2,42)
assert objective<.1 and len(np.unique(labels_test))==2
print('STEP 2: Compare RFM and LRFM (2–6 groups)', flush=True)
rows=[];profiles=[];assignments=m[['user_id']].copy()
for method,x in [('RFM',X[:,:3]),('LRFM',X)]:
 for k in range(2,7):
  print(f'  Fitting {method}, {k} groups', flush=True)
  lab,c,loss=fit(x,k,42)
  agreements=[]
  for seed in [43,44]:
   other,_,_=fit(x,k,seed);agreements.append(ari(lab,other))
  # Independent objective check using groupwise centered deviations.
  check=sum(float(((x[lab==j]-x[lab==j].mean(axis=0))**2).sum()) for j in range(k))
  assert abs(check-loss)<1e-5
  counts=np.bincount(lab,minlength=k)
  rows.append({'method':method,'k':k,'silhouette_sample1000':sil(x,lab),'stability_ari_min':min(agreements),'stability_ari_mean':float(np.mean(agreements)),'smallest_group':int(counts.min()),'largest_group':int(counts.max()),'inertia':loss})
  assignments[f'{method}_k{k}']=lab
  tmp=m.assign(cluster=lab)
  # Original-unit profiles + opportunity and perk-relevant features, with coverage.
  profile_cols=cols+['signup_tenure_days','recent_potential_exposure_days','flight_discounted_booking_rate','hotel_discounted_booking_rate','average_checked_bags','average_seats','average_known_hotel_nights','average_flight_lead_days','hotel_on_flight_trip_share','recent_hotel_trip_count','recent_flight_trip_count']
  for group,sub in tmp.groupby('cluster'):
   item={'method':method,'k':k,'cluster':int(group),'customers':len(sub)}
   item.update({c:float(sub[c].median()) if sub[c].notna().any() else None for c in profile_cols})
   item['has_discounted_valid_trip_share']=float(sub.discounted_valid_trip_count.gt(0).mean())
   item['has_cancellation_share']=float(sub.num_canceled_trips.gt(0).mean())
   profiles.append(item)
metrics=pd.DataFrame(rows);metrics.to_csv(ROOT/'rfm_lrfm_metrics.csv',index=False)
pd.DataFrame(profiles).to_csv(ROOT/'rfm_lrfm_profiles.csv',index=False);assignments.to_csv(ROOT/'rfm_lrfm_assignments.csv',index=False)

print("STEP 3: Profile the four-group RFM candidate", flush=True)
linked = d.merge(assignments, on="user_id", how="left", validate="one_to_one")
linked["candidate_group"] = linked["RFM_k4"].map(lambda v: f"RFM candidate {int(v) + 1}" if pd.notna(v) else "Outside comparison: " )
linked.loc[~ready, "candidate_group"] = "Outside comparison: " + linked.loc[~ready, "baseline_status"].astype(str)
linked.to_csv(ROOT / "all_customers_candidate_assignments.csv", index=False)
features = ["average_checked_bags", "average_seats", "average_known_hotel_nights",
            "flight_discounted_booking_rate", "hotel_discounted_booking_rate",
            "hotel_on_flight_trip_share", "average_flight_lead_days",
            "average_hotel_lead_days", "cancellation_rate", "net_booking_conversion",
            "recent_potential_exposure_days"]
behavior_rows = []
for group, sub in linked.loc[ready].groupby("candidate_group"):
    for feature in features:
        observed = pd.to_numeric(sub[feature], errors="coerce").dropna()
        behavior_rows.append({"candidate_group": group, "feature": feature,
                              "customers": len(sub), "customers_with_value": len(observed),
                              "coverage_share": len(observed) / len(sub),
                              "mean_customer_feature": observed.mean(),
                              "median_customer_feature": observed.median()})
pd.DataFrame(behavior_rows).to_csv(ROOT / "rfm4_behavior_profiles_with_coverage.csv", index=False)
coverage = d["baseline_status"].value_counts().rename_axis("status").reset_index(name="customers")
coverage.to_csv(ROOT / "comparison_population_coverage.csv", index=False)
assert len(linked) == len(d) and linked.user_id.is_unique
assert linked.loc[ready, "RFM_k4"].notna().all()
assert linked.loc[~ready, "RFM_k4"].isna().all()
report = ["# Preliminary RFM–LRFM Comparison and Candidate Profiles", "",
          f"Source: {source.name}. Full cohort: {len(d):,}; common modeling subset: {len(m):,}.",
          "", "## Method", "Log1p and population-standard-deviation scaling; identical eligible customers; NumPy Lloyd K-means with K-means++ initialization, 10 starts, up to 300 iterations. Main seed 42; sensitivity seeds 43 and 44. Silhouette uses a fixed 1,000-customer sample (seed 2026).",
          "", "## Internal validation", "```", metrics.round(4).to_string(index=False), "```",
          "", "## Four-group RFM candidate", "This is a profiling candidate, not a final model selection. The two-group alternatives remain baselines. Customer group numbers are arbitrary, not value rankings.",
          "```", pd.DataFrame([p for p in profiles if p["method"] == "RFM" and p["k"] == 4])[["cluster", "customers"] + cols].round(2).to_string(index=False), "```",
          "", "## Interpretation safeguards", "Frequency and Monetary use January 5–July 28, 2023; Recency and Length use available exported booking history. Travel profiles use the windows of their source features, generally full exported history, and are not relabeled as recent-window measures. Profile means of customer rates are unweighted customer averages, not pooled booking rates. Coverage is reported beside each feature. Missing spending is not replaced with zero. Customers outside the comparison remain explicit statuses, not final segments. Internal separation and seed stability do not prove perk effectiveness, loyalty, or profit.",
          "", "## Next decision", "Inspect behavior profiles for meaningful differences, assign justified perk hypotheses, and resolve full-cohort handling before selecting the final segmentation. Mention every final segment and highlight approximately three in the presentation."]
(ROOT / "candidate_profile_summary.md").write_text("\n".join(report), encoding="utf-8")
print("STEP 4 OK: Report and candidate profiles saved to", ROOT, flush=True)
print("Raw CSVs and the existing study feature table were not changed.", flush=True)
